https://github.com/UKPLab/sentence-transformers/blob/68dfbe643d51f1890e410b6783ca5343620db4fc/sentence_transformers/trainer.py#L620

In [ ]:

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset, DatasetDict

# Load a dataset (for example, IMDb for sentiment analysis)
imdb_dataset = load_dataset("imdb")
imdb_train_dataset = imdb_dataset['train'].shuffle().select(range(1))  # Small subset for quick training
imdb_eval_dataset = imdb_dataset['test'].shuffle().select(range(1))

ag_news_dataset = load_dataset("ag_news")
ag_news_train_dataset = ag_news_dataset['train'].shuffle().select(range(1))  # Small subset for quick training
ag_news_eval_dataset = ag_news_dataset['test'].shuffle().select(range(1))

# Load a pre-trained model and tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Tokenize the data
def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True)

imdb_train_dataset = imdb_train_dataset.map(tokenize, batched=True)
imdb_eval_dataset = imdb_eval_dataset.map(tokenize, batched=True)

ag_news_train_dataset = ag_news_train_dataset.map(tokenize, batched=True)
ag_news_eval_dataset = ag_news_eval_dataset.map(tokenize, batched=True)

'''
train_dataset = {
    "imdb_train_dataset": imdb_train_dataset,
    "ag_news_train_dataset": ag_news_train_dataset,
}

eval_dataset = {
    "imdb_eval_dataset": imdb_eval_dataset,
    "ag_news_eval_dataset": ag_news_eval_dataset,
}
'''

# Create a DatasetDict for training and evaluation
train_dataset = DatasetDict({
    "imdb": imdb_train_dataset,
    "ag_news": ag_news_train_dataset,
})

eval_dataset = DatasetDict({
    "imdb": imdb_eval_dataset,
    "ag_news": ag_news_eval_dataset,
})


# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=1,
    weight_decay=0.01,
    report_to = 'tensorboard'
)

#print(training_args.report_to)
#training_args.report_to = None

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print('Ready for training!')
# Train the model
trainer.train()


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset, DatasetDict
import torch
from torch import nn
from torch.utils.data import DataLoader

all_nli_pair_class_train = load_dataset("sentence-transformers/all-nli", "pair-class", split="train[:100]")

train_eval_split = all_nli_pair_class_train.train_test_split(test_size=0.2)

train_split_1 = train_eval_split['train'].train_test_split(test_size=0.01)['train']
train_split_2 = train_eval_split['train'].train_test_split(test_size=0.01)['test']

eval_split_1 = train_eval_split['test'].train_test_split(test_size=0.01)['train']
eval_split_2 = train_eval_split['test'].train_test_split(test_size=0.01)['test']

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

def tokenize_function(examples):
    return tokenizer(examples['premise'], examples['hypothesis'], padding="max_length", truncation=True)

train_split_1 = train_split_1.map(tokenize_function, batched=True)
train_split_2 = train_split_2.map(tokenize_function, batched=True)
eval_split_1 = eval_split_1.map(tokenize_function, batched=True)
eval_split_2 = eval_split_2.map(tokenize_function, batched=True)

train_dataset = DatasetDict({
    "imdb": train_split_1,
    "ag_news": train_split_2,
})

eval_dataset = DatasetDict({
    "imdb": eval_split_1,
    "ag_news": eval_split_2,
})

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    report_to="tensorboard",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=train_dataset,
)

print('Ready for training!')
trainer.train()
